# Test Generalisation of trained bert model on APP-350



**APP-350 corpus [Zimmeck et al. (2019)](https://doi.org/10.2478/popets-2019-0037)**: 350 app privacy policies.

**Annotations:** 53 privacy practice labels across 15,507 segments.

**Mapping:** Labels linked to MAPP-trained classifiers.

**Used for evaluation:** Only Contact, Location, and Device Identifier labels were confidently matched, giving 34 labels and 6,760 segments to test MAPP classifier generalisation.

## Import Libraries

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!pip install transformers datasets scikit-learn tqdm nltk joblib --quiet
import ast
import glob
import joblib
import numpy as np
import openpyxl
from openpyxl import load_workbook
import os
import pandas as pd
import pickle
import random
import re
import shutil
from collections import Counter
from pathlib import Path
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
tqdm.pandas()
import yaml
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          BertForSequenceClassification, BertTokenizerFast,
                          get_linear_schedule_with_warmup, logging as transformers_logging)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load APP-350 and Preprocess

In [4]:
folder_path = '/content/drive/MyDrive/Colab Notebooks/Project/annotations/'

all_annotations = []
total_segments_overall = 0
total_files = 0

yaml_files = [f for f in os.listdir(folder_path) if f.endswith(('.yaml', '.yml'))]

# progress bar
for filename in tqdm(yaml_files, desc="Processing YAML files"):
    file_path = os.path.join(folder_path, filename)
    with open(file_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

        total_files += 1
        policy_id = data.get('policy_id', '')
        policy_name = data.get('policy_name', '')

        segments = data.get('segments', [])
        total_segments_overall += len(segments)

        tqdm.write(f"{filename} ({policy_name}): {len(segments)} segments")

        for segment in segments:
            segment_id = segment.get('segment_id', '')
            segment_text = segment.get('segment_text', '')

            annotations = segment.get('annotations', [])
            for ann in annotations:
                practice = ann.get('practice', '')
                modality = ann.get('modality', '')

                all_annotations.append({
                    'policy_id': policy_id,
                    'policy_name': policy_name,
                    'segment_id': segment_id,
                    'segment_text': segment_text,
                    'practice': practice,
                    'modality': modality
                })

df = pd.DataFrame(all_annotations)
df.to_csv('annotations.csv', index=False)


Processing YAML files:   0%|          | 0/350 [00:00<?, ?it/s]

policy_247.yml (JiubangDigitalTechnologyZcamera): 14 segments
policy_348.yml (Zedge): 40 segments
policy_347.yml (YouVersion): 19 segments
policy_349.yml (Zello): 14 segments
policy_350.yml (Zynga): 102 segments
policy_105.yml (Alibaba): 48 segments
policy_130.yml (CheetahMobileCMLauncher): 18 segments
policy_103.yml (AceViral): 7 segments
policy_114.yml (AxesInMotion): 32 segments
policy_110.yml (Asus): 90 segments
policy_142.yml (com.arcsoft.perfect365): 42 segments
policy_126.yml (bzing): 6 segments
policy_150.yml (com.celltick.lockscreen): 31 segments
policy_120.yml (Bestcoolfungames): 13 segments
policy_121.yml (BigDuckGames): 12 segments
policy_138.yml (com.appgeneration.itunerfree): 22 segments
policy_124.yml (BubbleQuizGames): 59 segments
policy_147.yml (com.bxapps.CleanHouseForKids): 7 segments
policy_117.yml (Baidu): 57 segments
policy_98.yml (Xender): 20 segments
policy_81.yml (SoloLauncher): 20 segments
policy_3.yml (AppliqatoSoftware): 8 segments
policy_5.yml (BarcodeScann

In [5]:
print(f"Total files processed: {total_files}")
print(f"Total segments across all files: {total_segments_overall}")
print(f"Total annotations extracted: {len(all_annotations)}")
print(df.shape)
print(" ")
print(" all label counts")
print(len(df['practice'].value_counts()))

#print(df['practice'].value_counts())


Total files processed: 350
Total segments across all files: 15507
Total annotations extracted: 10201
(10201, 6)
 
 all label counts
58


In [6]:
df.columns

Index(['policy_id', 'policy_name', 'segment_id', 'segment_text', 'practice',
       'modality'],
      dtype='object')

In [7]:
# mappings
safe_mapping = {
    # IP address and device IDs
    "Identifier_IP_Address_1stParty": "Information Type_IP address and device IDs",
    "Identifier_IP_Address_3rdParty": "Information Type_IP address and device IDs",
    "Identifier_Device_ID_1stParty": "Information Type_IP address and device IDs",
    "Identifier_Device_ID_3rdParty": "Information Type_IP address and device IDs",
    "Identifier_MAC_1stParty": "Information Type_IP address and device IDs",
    "Identifier_MAC_3rdParty": "Information Type_IP address and device IDs",
    "Identifier_IMEI_1stParty": "Information Type_IP address and device IDs",
    "Identifier_IMEI_3rdParty": "Information Type_IP address and device IDs",
    "Identifier_Ad_ID_1stParty": "Information Type_IP address and device IDs",
    "Identifier_Ad_ID_3rdParty": "Information Type_IP address and device IDs",

    # Contact information
    "Contact_E_Mail_Address_1stParty": "Information Type_Contact information",
    "Contact_E_Mail_Address_3rdParty": "Information Type_Contact information",
    "Contact_Phone_Number_1stParty": "Information Type_Contact information",
    "Contact_Phone_Number_3rdParty": "Information Type_Contact information",
    "Contact_Postal_Address_1stParty": "Information Type_Contact information",
    "Contact_Postal_Address_3rdParty": "Information Type_Contact information",
    "Contact_ZIP_1stParty": "Information Type_Contact information",
    "Contact_ZIP_3rdParty": "Information Type_Contact information",
    "Contact_City_1stParty": "Information Type_Contact information",
    "Contact_City_3rdParty": "Information Type_Contact information",
    "Contact_Address_Book_1stParty": "Information Type_Contact information",
    "Contact_Address_Book_3rdParty": "Information Type_Contact information",


    # Location
    "Location_1stParty": "Information Type_Location",
    "Location_3rdParty": "Information Type_Location",
    "Location_GPS_1stParty": "Information Type_Location",
    "Location_GPS_3rdParty": "Information Type_Location",
    "Location_WiFi_1stParty": "Information Type_Location",
    "Location_WiFi_3rdParty": "Information Type_Location",
    "Location_Cell_Tower_1stParty": "Information Type_Location",
    "Location_Cell_Tower_3rdParty": "Information Type_Location",
    "Location_Bluetooth_1stParty": "Information Type_Location",
    "Location_Bluetooth_3rdParty": "Information Type_Location",
    "Location_IP_Address_1stParty": "Information Type_Location",
    "Location_IP_Address_3rdParty": "Information Type_Location",

}

# keep rows mapped
df_safe = df[df["practice"].isin(safe_mapping.keys())].copy()

#add new column for new
df_safe["Mapped_Label"] = df_safe["practice"].map(safe_mapping)

# Show df
print(f"Total annotations extracted: {len(all_annotations)}")
print(f"Total rows before: {len(df)}")
print(f"Rows after safe filter: {len(df_safe)}")
print(df_safe["Mapped_Label"].value_counts())

Total annotations extracted: 10201
Total rows before: 10201
Rows after safe filter: 6760
Mapped_Label
Information Type_Contact information          2731
Information Type_Location                     2055
Information Type_IP address and device IDs    1974
Name: count, dtype: int64


## Test

In [8]:
# use bert set up
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/models/win")
TEST_EXCEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Project/test_app350_bert.xlsx"
BATCH_SIZE = 32
MAX_LEN = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Only labels in app350 filtered dataset
custom_labels = [
    'Information Type_Contact information',
    'Information Type_Location',
    'Information Type_IP address and device IDs'
]

# convert text and labels into token IDs, attention masks, and tensors for training
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Evaluation
def evaluate_model(model, dataloader, label_name):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {label_name}", leave=True):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits).squeeze(-1)
            probs = probs.detach().cpu().numpy().flatten()
            preds.extend(probs)

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall, bin_preds, true_labels


# Test
results = []
all_labels_preds = {}
for label_to_test in tqdm(custom_labels, desc="Testing all labels"):
    safe_label = label_to_test.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_bert_model_{safe_label}.pt"

    if not model_path.exists():
        print(f"Model file missing for {label_to_test}, skipping.")
        continue

    print(f"\nEvaluating model for label: {label_to_test}")

    # test data
    df_safe['Test_Label'] = df_safe['Mapped_Label'].copy()

    df_safe['binary_label'] = df_safe['Test_Label'].apply(lambda x: 1 if x == label_to_test else 0)

    test_dataset = TextDataset(df_safe['segment_text'].tolist(), df_safe['binary_label'].tolist())
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    # Load model
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))

    acc, f1, precision, recall, bin_preds, true_labels = evaluate_model(model, test_loader, label_to_test)
    print(f"Results: Acc={acc:.4f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}\n")

    all_labels_preds[label_to_test] = {
        'true': true_labels,
        'pred': bin_preds
    }

    results.append({
        'label': label_to_test,
        'accuracy': round(acc, 4),
        'f1_score': round(f1, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
    })

# ==== Save results ====
app350_df = pd.DataFrame(results)
app350_df.to_excel(TEST_EXCEL_PATH, index=False)
print(f"\nTest results saved to: {TEST_EXCEL_PATH}")

app350_df

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Results: Acc=0.5956, F1=0.5455, Precision=0.4143, Recall=0.7985


Evaluating model for label: Information Type_IP address and device IDs


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating Information Type_IP address and device IDs:   0%|          | 0/212 [00:00<?, ?it/s]

Results: Acc=0.7629, F1=0.6498, Precision=0.5713, Recall=0.7533


Test results saved to: /content/drive/MyDrive/Colab Notebooks/Project/test_app350_bert.xlsx


,label,accuracy,f1_score,precision,recall
0,Information Type_Contact information,0.8072,0.7694,0.7445,0.7960
1,Information Type_Location,0.5956,0.5455,0.4143,0.7985
2,Information Type_IP address and device IDs,0.7629,0.6498,0.5713,0.7533
